# Query Plans

[`util:explain()`]({docs}/functions/util/explain) shows you what the XQuery optimizer produces — the compiled expression tree that actually executes. This is the key to understanding why a query is fast or slow.

## Basic Expression Tree

Pass a query as a string to see its compiled form:

In [ ]:
util:explain('1 + 1')

## FLWOR Expressions

The optimizer breaks FLWOR expressions into their component clauses:

In [ ]:
util:explain('
    for $x in 1 to 10
    where $x > 5
    return $x * 2
')

Notice the `<for>`, `<in>`, and nested expressions. The `where` clause becomes a comparison node.

## Let Bindings

In [ ]:
util:explain('
    let $greeting := "hello"
    let $name := "world"
    return concat($greeting, " ", $name)
')

## Path Expressions

Path expressions show the axis and node test for each step:

In [ ]:
util:explain('//book[@year > 2020]/title')

Look for `<step axis="..." test="...">` elements — these show how the engine traverses the document tree.

## Predicates

Predicates appear as children of their step:

In [ ]:
util:explain('//section[position() < 5][title]')

## Function Calls

See how function calls are compiled — built-in functions may appear differently from user-defined ones:

In [ ]:
util:explain('
    let $nums := 1 to 100
    return (count($nums), sum($nums), avg($nums))
')

## Conditionals

In [ ]:
util:explain('
    if (true()) then
        "yes"
    else
        "no"
')

## Nested FLWOR with Grouping

Complex queries produce deeper trees:

In [ ]:
util:explain('
    for $x in 1 to 20
    let $group := $x mod 3
    group by $group
    order by $group
    return map { "group": $group, "values": array { $x } }
')

## Reading the Tree

Tips for interpreting the explain output:

- **`<for>`** / **`<let>`** — FLWOR clause with `@variable` showing the binding
- **`<step>`** — path step with `@axis` (child, descendant, attribute, etc.) and `@test` (element name, `node()`, `*`)
- **`<predicate>`** — filter expression inside `[...]`
- **`<comparison>`** — value or general comparison
- **`<function-call>`** / **`<builtin-function>`** — function invocation with `@name` and `@arity`
- **`<if>`** — conditional with `<test>`, `<then>`, `<else>` children
- **`@line`** / **`@column`** — source location (when available)

## Comparing Two Approaches

Use explain to understand WHY one approach might be faster:

In [ ]:
<comparison>
    <approach label="general-comparison">
        { util:explain('//book[@year = 2020]') }
    </approach>
    <approach label="value-comparison">
        { util:explain('//book[@year eq 2020]') }
    </approach>
</comparison>